# LAB 10 — K-Means Clustering
## Student Information
- Name:
- Student ID:
- Section:
- GitHub:
- Kaggle:
- Dataset: Mall Customers (Kaggle) + Wine (UCI)

**Dataset file:** Upload `Mall_Customers.csv` to the same Colab/Jupyter working folder before running.

**Rule:** This is unsupervised learning. There is no target column, so do not use train/test splitting or accuracy. Use `SEED = 42`.

In [ ]:
# Imports and setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples, adjusted_rand_score
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_wine
from sklearn.decomposition import PCA

SEED = 42
plt.rcParams['figure.figsize'] = (7, 4)

# Part 0 — Setup and First Look
## Task 0.1 — Load and check the data
Load the CSV, shorten the column names, inspect shape, missing values, duplicates, gender counts and descriptive statistics.

In [ ]:
df = pd.read_csv('Mall_Customers.csv')
df = df.rename(columns={
    'Annual Income (k$)': 'Income',
    'Spending Score (1-100)': 'Spending'
})

print('Shape:', df.shape)
print('Missing values per column:\n', df.isna().sum())
print('Duplicate rows:', df.duplicated().sum())
print('Gender counts:\n', df['Gender'].value_counts())
display(df[['Age', 'Income', 'Spending']].describe())

### Task 0.2 — Look at the data before clustering
The three scatter plots let us inspect possible groupings before applying K-Means.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
axes[0].scatter(df['Income'], df['Spending'])
axes[0].set(xlabel='Annual Income (k$)', ylabel='Spending Score', title='Spending vs Income')
axes[1].scatter(df['Age'], df['Spending'])
axes[1].set(xlabel='Age', ylabel='Spending Score', title='Spending vs Age')
axes[2].scatter(df['Age'], df['Income'])
axes[2].set(xlabel='Age', ylabel='Annual Income (k$)', title='Income vs Age')
plt.tight_layout()
plt.show()

**Markdown answers — Part 0**
- Spending vs Income shows about five visible patterns most clearly: low-income/high-spending, low-income/low-spending, middle-income/middle-spending, high-income/high-spending and high-income/low-spending.
- Spending vs Age and Income vs Age show weaker/more overlapping patterns; the exact visual count is approximate.
- Accuracy and train/test splitting are unsuitable because there is no known target label or correct class for each customer. We evaluate clusters using distances, inertia, silhouette, plots and domain sense.

# Part 1 — K-Means From Scratch
## Shared function: Euclidean distances and custom K-Means
The function below does not call `KMeans`. It assigns each point to the nearest centroid, updates each centroid to its cluster mean, keeps an old centroid if its cluster becomes empty, and returns labels, centroids, inertia and iteration count.

In [ ]:
def my_kmeans(X, init_centroids, max_iter=100):
    X = np.asarray(X, dtype=float)
    centroids = np.asarray(init_centroids, dtype=float).copy()

    for iteration in range(1, max_iter + 1):
        distances = np.sqrt(((X[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2))
        labels = distances.argmin(axis=1)

        new_centroids = centroids.copy()
        for k in range(len(centroids)):
            if np.any(labels == k):
                new_centroids[k] = X[labels == k].mean(axis=0)
            # Empty cluster: keep its old centroid.

        if np.allclose(new_centroids, centroids):
            centroids = new_centroids
            break
        centroids = new_centroids

    distances = np.sqrt(((X[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2))
    labels = distances.argmin(axis=1)
    inertia = ((X - centroids[labels]) ** 2).sum()
    return labels, centroids, inertia, iteration

### Task 1.1 — Lecture Example 1 (one feature)
Data: 2, 4, 10, 12, 3, 20, 30, 11, 25. K = 2; initial centroids = 4 and 11.

In [ ]:
x1 = np.array([2, 4, 10, 12, 3, 20, 30, 11, 25], dtype=float).reshape(-1, 1)
centroids = np.array([[4.], [11.]])
for iteration in range(1, 20):
    distances = np.abs(x1 - centroids.T)
    labels = distances.argmin(axis=1)
    new_centroids = centroids.copy()
    for k in range(2):
        if np.any(labels == k):
            new_centroids[k] = x1[labels == k].mean(axis=0)
    print(f'Iteration {iteration}')
    for k in range(2):
        print(f'  C{k+1}: {x1[labels == k].ravel().tolist()}')
    print('  New centroids:', new_centroids.ravel())
    if np.allclose(new_centroids, centroids):
        centroids = new_centroids
        break
    centroids = new_centroids

labels1, final_c1, inertia1, it1 = my_kmeans(x1, [[4], [11]])
print('Final centroids:', final_c1.ravel())
print('Inertia:', inertia1, '| Iterations:', it1)

**Task 1.1 interpretation:** Final clusters are `{2, 4, 10, 12, 3, 11}` and `{20, 30, 25}`. Final centroids are 7 and 25; inertia is 150, matching the lecture.

### Task 1.2 — Lecture Example 2 (two features)
Height/weight points P1–P6; K = 2; initial centroids are P1 and P2.

In [ ]:
Xh = np.array([[185,72], [170,56], [168,60], [179,68], [182,72], [188,77]], dtype=float)
Ch = Xh[[0, 1]].copy()
for iteration in range(1, 20):
    D = np.sqrt(((Xh[:, None, :] - Ch[None, :, :]) ** 2).sum(axis=2))
    labels_h = D.argmin(axis=1)
    new_Ch = Ch.copy()
    for k in range(2):
        if np.any(labels_h == k):
            new_Ch[k] = Xh[labels_h == k].mean(axis=0)
    print(f'Iteration {iteration}:')
    print('  Distances (rows P1-P6, columns M1-M2):\n', np.round(D, 2))
    print('  Cluster 1:', [f'P{i+1}' for i in range(6) if labels_h[i] == 0])
    print('  Cluster 2:', [f'P{i+1}' for i in range(6) if labels_h[i] == 1])
    print('  New centroids:\n', new_Ch)
    if np.allclose(new_Ch, Ch):
        Ch = new_Ch
        break
    Ch = new_Ch
print('Final centroids:\n', Ch)
print('Check d(P3, M2) in first iteration =', np.linalg.norm(Xh[2] - Xh[1]))

**Task 1.2 interpretation:** Cluster 1 = P1, P4, P5, P6; Cluster 2 = P2, P3. Final centroids are (183.5, 72.25) and (169, 58). The first-iteration distance d(P3, M2) is √20 = 4.47.

### Task 1.3 — Write and test your own K-Means function

In [ ]:
labels_custom, centers_custom, inertia_custom, iterations_custom = my_kmeans(x1, [[4], [11]])
print('Labels:', labels_custom)
print('Centroids:', centers_custom.ravel())
print('Inertia:', inertia_custom)
print('Iterations:', iterations_custom)
# Expected: centroids [7, 25], inertia 150.

### Task 1.4 — Compare with scikit-learn
Use Income and Spending (200 × 2); first five rows are initial centroids.

In [ ]:
X2 = df[['Income', 'Spending']].to_numpy()
initial_5 = X2[:5].copy()

my_labels, my_centers, my_inertia, my_iterations = my_kmeans(X2, initial_5)
sk_model = KMeans(
    n_clusters=5, init=initial_5, n_init=1,
    tol=0, algorithm='lloyd', random_state=SEED
).fit(X2)

print('Custom iterations:', my_iterations, '| sklearn iterations:', sk_model.n_iter_)
print('Custom inertia:', my_inertia, '| sklearn inertia:', sk_model.inertia_)
print('Labels exactly equal:', np.array_equal(my_labels, sk_model.labels_))
print('Centroids equal:', np.allclose(my_centers, sk_model.cluster_centers_))
print('Custom centroids:\n', my_centers)
print('sklearn centroids:\n', sk_model.cluster_centers_)

**Markdown answers — Part 1**
- (a) Point 10 begins in C2. After iteration 1 the centroids are 3 and 18. Its distances are |10−3| = 7 and |10−18| = 8, so it moves to C1 in iteration 2.
- (b) The loop stops when centroids no longer change (or the iteration limit is reached). Assign/update reduces or preserves within-cluster squared distance, and only finitely many assignments exist, so it reaches a stable assignment. A stable result can still be a local minimum, not the globally best solution.
- (c) With the same initial centroids and Lloyd settings, the custom function should match sklearn's labels, centroids and inertia (allowing tiny floating-point differences).
- (d) A centroid is the average Income and Spending of the customers in a cluster. K-Means requires numeric features because it calculates means and distances.

# Part 2 — Choosing K: Elbow and Silhouette
## Task 2.1 — Elbow and silhouette for K = 2 to 10

In [ ]:
results = []
models_k = {}
for k in range(2, 11):
    model = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X2)
    models_k[k] = model
    results.append({
        'K': k,
        'Inertia': model.inertia_,
        'Silhouette': silhouette_score(X2, model.labels_)
    })
results_df = pd.DataFrame(results)
display(results_df.round(4))
print('Highest silhouette K:', int(results_df.loc[results_df['Silhouette'].idxmax(), 'K']))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(results_df['K'], results_df['Inertia'], marker='o')
axes[0].set(xlabel='K', ylabel='Inertia', title='Elbow Method')
axes[1].bar(results_df['K'], results_df['Silhouette'])
axes[1].set(xlabel='K', ylabel='Average silhouette', title='Silhouette by K')
plt.tight_layout()
plt.show()

### Task 2.2 — Silhouette by hand
For a point: `a` is its average distance to points in its own cluster; `b` is the smallest average distance to another cluster; `s = (b-a)/max(a,b)`. 

In [ ]:
def hand_silhouette(X, labels, index):
    point = X[index]
    own_cluster = labels[index]
    same = X[labels == own_cluster]
    same_distances = np.linalg.norm(same - point, axis=1)
    same_distances = same_distances[same_distances > 0]
    a = same_distances.mean()

    other_means = []
    for cluster in np.unique(labels):
        if cluster != own_cluster:
            other = X[labels == cluster]
            other_means.append(np.linalg.norm(other - point, axis=1).mean())
    b = min(other_means)
    s = (b - a) / max(a, b)
    return a, b, s

model5 = KMeans(n_clusters=5, n_init=10, random_state=SEED).fit(X2)
sample_sil = silhouette_samples(X2, model5.labels_)
a0, b0, s0 = hand_silhouette(X2, model5.labels_, 0)
print('Customer 1: a =', a0, 'b =', b0, 's =', s0)
print('sklearn silhouette:', sample_sil[0])
sil_df = pd.DataFrame({'Cluster': model5.labels_, 'Silhouette': sample_sil})
display(sil_df.groupby('Cluster')['Silhouette'].agg(['count', 'mean', 'min']).round(4))
print('Negative silhouette customers:', int((sample_sil < 0).sum()))

# Silhouette plot, sorted within each cluster
fig, ax = plt.subplots(figsize=(8, 6))
y_lower = 10
for cluster in range(5):
    vals = np.sort(sample_sil[model5.labels_ == cluster])
    y_upper = y_lower + len(vals)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, vals)
    ax.text(-0.05, y_lower + len(vals)/2, str(cluster))
    y_lower = y_upper + 10
ax.axvline(sample_sil.mean(), linestyle='--')
ax.set(xlabel='Silhouette coefficient', ylabel='Customers (sorted within cluster)', title='Silhouette Plot — K=5')
plt.show()

### Task 2.3 — Lecture practice problem
Points = 1, 3, 5, 7, 11, 13, 15; K = 2; starting centroids 1 and 11.

In [ ]:
practice = np.array([1, 3, 5, 7, 11, 13, 15], dtype=float).reshape(-1, 1)
p_labels, p_centers, p_inertia, p_iterations = my_kmeans(practice, [[1], [11]])
print('Cluster 1:', practice[p_labels == 0].ravel())
print('Cluster 2:', practice[p_labels == 1].ravel())
print('Centroids:', p_centers.ravel(), '| Inertia:', p_inertia, '| Iterations:', p_iterations)
for value in [1, 7, 11]:
    idx = int(np.where(practice.ravel() == value)[0][0])
    print(value, 'hand (a,b,s):', hand_silhouette(practice, p_labels, idx))
print('sklearn samples:', silhouette_samples(practice, p_labels))

**Markdown answers — Part 2**
- (a) The highest silhouette is K=5 (about 0.5539). The elbow also bends strongly around K=5: inertia drops by about 29,231.3 from K=4 to 5, then by about 7,214.6 from K=5 to 6. Both support K=5.
- (b) Customer 1 has silhouette about 0.4170, indicating a reasonably placed point, but not an extremely strong fit. Cluster 3 has the weakest mean silhouette (about 0.5040); 2 customers have negative silhouettes.
- (c) Point 7 is less well placed than point 1: point 1 has s≈0.667, while point 7 has s≈0.333. For point 7, a=4 and b=6, so it is closer to the boundary.
- (d) Inertia always decreases as K increases and becomes zero if every point is its own cluster. Lowest inertia therefore trivially favors too many clusters.

# Part 3 — Starting Points and Stability
## Task 3.1 — 30 different seeds

In [ ]:
stability = {}
summary = []
for setting, init_method, n_init in [
    ('random, n_init=1', 'random', 1),
    ('k-means++, n_init=1', 'k-means++', 1),
    ('k-means++, n_init=10', 'k-means++', 10)
]:
    inertias = []
    fitted = []
    for seed in range(30):
        model = KMeans(n_clusters=5, init=init_method, n_init=n_init, random_state=seed).fit(X2)
        inertias.append(model.inertia_)
        fitted.append(model)
    stability[setting] = (inertias, fitted)
    best = min(inertias)
    summary.append({
        'Setting': setting,
        'Distinct inertia values (1 dp)': len(set(np.round(inertias, 1))),
        'Lowest': min(inertias),
        'Highest': max(inertias),
        '% reaching best': 100 * np.isclose(inertias, best).sum() / len(inertias)
    })
display(pd.DataFrame(summary).round(2))

plt.figure(figsize=(8, 4))
for setting, (inertias, _) in stability.items():
    plt.hist(inertias, bins=12, alpha=0.45, label=setting)
plt.xlabel('Final inertia')
plt.ylabel('Number of runs')
plt.title('Final Inertia Across 30 Seeds')
plt.legend()
plt.show()

### Task 3.2 — Best run vs worst run

In [ ]:
random_inertias, random_models = stability['random, n_init=1']
best_i = int(np.argmin(random_inertias))
worst_i = int(np.argmax(random_inertias))
best_model = random_models[best_i]
worst_model = random_models[worst_i]
print('Best inertia:', best_model.inertia_)
print('Worst inertia:', worst_model.inertia_)
print('ARI between best and worst:', adjusted_rand_score(best_model.labels_, worst_model.labels_))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, model, title in [
    (axes[0], best_model, 'Best random-init run'),
    (axes[1], worst_model, 'Worst random-init run')
]:
    ax.scatter(X2[:, 0], X2[:, 1], c=model.labels_)
    ax.scatter(model.cluster_centers_[:, 0], model.cluster_centers_[:, 1], marker='X', s=180)
    ax.set(xlabel='Income (k$)', ylabel='Spending', title=f'{title}\nInertia={model.inertia_:.1f}')
plt.tight_layout()
plt.show()

**Markdown answers — Part 3**
- (a) In this run, random initialization produced 14 distinct inertia values and reached the best solution in 33.3% of runs. k-means++ with n_init=1 produced 6 values and reached it in 60%; k-means++ with n_init=10 produced 1 value and reached it in 100%.
- (b) Different starting centroids can lead the assign/update process into different local minima.
- (c) k-means++ spreads initial centroids more sensibly; n_init repeats the process and keeps the lowest-inertia result. Here n_init=10 consistently found the best solution.
- (d) Cluster numbers are arbitrary. ARI compares membership agreement while ignoring whether a group is called cluster 0 or cluster 3.

# Part 4 — Scaling: Units Decide the Clusters
## Task 4.1 — Change the unit, change the clusters

In [ ]:
age_income = df[['Age', 'Income']].to_numpy()
income_dollars = age_income.copy()
income_dollars[:, 1] *= 1000
scaled_k = StandardScaler().fit_transform(age_income)
scaled_dollars = StandardScaler().fit_transform(income_dollars)

unit_models = {}
for name, data in [
    ('Raw k$', age_income),
    ('Raw dollars', income_dollars),
    ('Scaled k$', scaled_k),
    ('Scaled dollars', scaled_dollars)
]:
    unit_models[name] = KMeans(n_clusters=4, n_init=10, random_state=SEED).fit(data)

print('ARI raw k$ vs raw dollars:', adjusted_rand_score(unit_models['Raw k$'].labels_, unit_models['Raw dollars'].labels_))
print('ARI scaled k$ vs scaled dollars:', adjusted_rand_score(unit_models['Scaled k$'].labels_, unit_models['Scaled dollars'].labels_))
print('ARI raw dollars vs scaled dollars:', adjusted_rand_score(unit_models['Raw dollars'].labels_, unit_models['Scaled dollars'].labels_))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, name in zip(axes, ['Raw k$', 'Raw dollars', 'Scaled k$']):
    ax.scatter(df['Age'], df['Income'], c=unit_models[name].labels_)
    ax.set(xlabel='Age', ylabel='Income (k$)', title=name)
plt.tight_layout()
plt.show()

### Task 4.2 — Wine data: raw vs scaled

In [ ]:
wine = load_wine()
Xw, y_true = wine.data, wine.target
feature_std = pd.Series(Xw.std(axis=0), index=wine.feature_names).sort_values(ascending=False)
print('Three largest standard deviations:\n', feature_std.head(3))
print('Smallest standard deviation:\n', feature_std.tail(1))

Xw_scaled = StandardScaler().fit_transform(Xw)
wine_raw_model = KMeans(n_clusters=3, n_init=10, random_state=SEED).fit(Xw)
wine_scaled_model = KMeans(n_clusters=3, n_init=10, random_state=SEED).fit(Xw_scaled)

for name, Xdata, model in [
    ('Raw', Xw, wine_raw_model),
    ('Scaled', Xw_scaled, wine_scaled_model)
]:
    print(name, '| ARI:', adjusted_rand_score(y_true, model.labels_),
          '| Silhouette:', silhouette_score(Xdata, model.labels_),
          '| Cluster sizes:', np.bincount(model.labels_))

### Task 4.3 — PCA comparison

In [ ]:
pca = PCA(n_components=2, random_state=SEED)
wine_2d = pca.fit_transform(Xw_scaled)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, labels, title in [
    (axes[0], y_true, 'True cultivars'),
    (axes[1], wine_raw_model.labels_, 'Raw-data K-Means'),
    (axes[2], wine_scaled_model.labels_, 'Scaled-data K-Means')
]:
    ax.scatter(wine_2d[:, 0], wine_2d[:, 1], c=labels)
    ax.set(xlabel='PC1', ylabel='PC2', title=title)
plt.tight_layout()
plt.show()

**Markdown answers — Part 4**
- (a) Changing Income from thousands to dollars multiplies that feature's numeric differences by 1000. Squared Euclidean distance then gives Income vastly more influence than Age. Scaling restores comparable feature units.
- (b) Proline dominates raw Wine distances: its standard deviation is about 314.02, much larger than the next feature (Magnesium, about 14.24). ARI is about 0.3711 raw and 0.8975 scaled.
- (c) Raw silhouette is higher (about 0.5711 vs 0.2849), but that only measures compactness/separation under each feature space's own distances. Raw distances are dominated by Proline, so the clusters can be geometrically separated while disagreeing with true cultivars. Silhouettes from differently scaled feature spaces are not directly comparable.
- (d) Scale when features represent comparable dimensions but use different numeric units or ranges. Do not automatically scale if the original units/weights intentionally represent business importance; decide based on the meaning of the features.

# Part 5 — Outliers and Anomalies
## Task 5.1 — One extreme customer

In [ ]:
X_extra = np.vstack([X2, [400, 99]])
original_model = KMeans(n_clusters=5, n_init=10, random_state=SEED).fit(X2)
extra_model = KMeans(n_clusters=5, n_init=10, random_state=SEED).fit(X_extra)

print('Original cluster sizes:', np.bincount(original_model.labels_))
print('With extra customer sizes:', np.bincount(extra_model.labels_))
print('Original centroids sorted by income:\n', original_model.cluster_centers_[np.argsort(original_model.cluster_centers_[:, 0])])
print('New centroids sorted by income:\n', extra_model.cluster_centers_[np.argsort(extra_model.cluster_centers_[:, 0])])
print('ARI on original 200 customers:', adjusted_rand_score(original_model.labels_, extra_model.labels_[:200]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, labels, centers, title in [
    (axes[0], original_model.labels_, original_model.cluster_centers_, 'Original 200 customers'),
    (axes[1], extra_model.labels_[:200], extra_model.cluster_centers_, 'With extreme customer')
]:
    ax.scatter(X2[:, 0], X2[:, 1], c=labels)
    ax.scatter(centers[:, 0], centers[:, 1], marker='X', s=180)
    ax.set(xlabel='Income (k$)', ylabel='Spending', title=title)
axes[1].scatter([400], [99], marker='*', s=220, label='Extra customer')
axes[1].legend()
plt.tight_layout()
plt.show()

### Task 5.2 — Catch it and repair

In [ ]:
# Distance from each original customer to its assigned centroid
real_distances = np.linalg.norm(X2 - original_model.cluster_centers_[original_model.labels_], axis=1)
extra_distance = np.linalg.norm(np.array([400, 99]) - original_model.cluster_centers_, axis=1).min()
threshold = real_distances.mean() + 3 * real_distances.std()
flagged = real_distances > threshold

print('Mean distance:', real_distances.mean())
print('Std distance:', real_distances.std())
print('Threshold (mean + 3 std):', threshold)
print('Real customers flagged:', int(flagged.sum()))
print('Extra customer distance:', extra_distance, '| Flagged:', extra_distance > threshold)

top5_idx = np.argsort(real_distances)[-5:][::-1]
top5 = df.iloc[top5_idx][['CustomerID', 'Age', 'Income', 'Spending']].copy()
top5['Cluster'] = original_model.labels_[top5_idx]
top5['Distance'] = real_distances[top5_idx]
display(top5)

# Remove flagged rows, refit, then predict all original 200 customers
clean_df = df.loc[~flagged].copy()
repair_model = KMeans(n_clusters=5, n_init=10, random_state=SEED).fit(clean_df[['Income', 'Spending']])
repaired_predictions = repair_model.predict(X2)
print('Repaired cluster sizes:', np.bincount(repair_model.labels_))
print('ARI vs original model on all 200:', adjusted_rand_score(original_model.labels_, repaired_predictions))

**Markdown answers — Part 5**
- (a) The extreme point creates its own singleton cluster and changes the remaining grouping. The original clusters of 81 and 35 customers are especially affected/redistributed; compare the printed centroid tables and ARI.
- (b) The distance rule flags the added customer because its nearest-centroid distance is far above the threshold. It also flags four real high-income customers (IDs 197–200 in this dataset). They are not necessarily errors; they may be unusual but genuine customers worth investigating.
- (c) Two approaches: investigate/correct data-entry errors, or use robust methods/transformations/capping where justified. Deleting outliers can remove valid rare customers and distort the business picture.

# Part 6 — From Clusters to Customer Segments
## Task 6.1 — Profile and name the segments

In [ ]:
segment_model = KMeans(n_clusters=5, n_init=10, random_state=SEED).fit(X2)
seg_df = df.copy()
seg_df['Cluster'] = segment_model.labels_
profile = seg_df.groupby('Cluster').agg(
    Customers=('CustomerID', 'size'),
    Age=('Age', 'mean'),
    Income=('Income', 'mean'),
    Spending=('Spending', 'mean'),
    Female_pct=('Gender', lambda x: 100 * (x == 'Female').mean())
).sort_values('Income')
display(profile.round(2))

def make_segment(row):
    income_level = 'Low' if row['Income'] < 40 else ('High' if row['Income'] >= 70 else 'Mid')
    spending_level = 'Low' if row['Spending'] < 35 else ('High' if row['Spending'] >= 65 else 'Mid')
    return f'{income_level} Income / {spending_level} Spending'

cluster_means = seg_df.groupby('Cluster')[['Income', 'Spending']].mean()
cluster_to_segment = {cluster: make_segment(row) for cluster, row in cluster_means.iterrows()}
seg_df['Segment'] = seg_df['Cluster'].map(cluster_to_segment)
print('Segment sizes:\n', seg_df['Segment'].value_counts())

### Task 6.2 — Does Gender matter? Does Age matter?

In [ ]:
gender_by_segment = pd.crosstab(seg_df['Segment'], seg_df['Gender'], normalize='index') * 100
print('Gender percentages within segment:')
display(gender_by_segment.round(2))
print('Overall female percentage:', 100 * (df['Gender'] == 'Female').mean())

X3 = StandardScaler().fit_transform(df[['Age', 'Income', 'Spending']])
sil_age_income_spend = []
for k in range(2, 9):
    model = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X3)
    sil_age_income_spend.append((k, silhouette_score(X3, model.labels_)))
display(pd.DataFrame(sil_age_income_spend, columns=['K', 'Silhouette']).round(4))

### Task 6.3 — A three-feature alternative

In [ ]:
best_k_3features = max(sil_age_income_spend, key=lambda x: x[1])[0]
three_feature_model = KMeans(n_clusters=best_k_3features, n_init=10, random_state=SEED).fit(X3)
three_profile = df[['Age', 'Income', 'Spending']].copy()
three_profile['Cluster'] = three_feature_model.labels_
display(three_profile.groupby('Cluster').agg(
    Customers=('Age', 'size'),
    Age=('Age', 'mean'),
    Income=('Income', 'mean'),
    Spending=('Spending', 'mean')
).sort_values('Age').round(2))
print('Best K for three scaled features:', best_k_3features)
print('ARI vs two-feature K=5:', adjusted_rand_score(segment_model.labels_, three_feature_model.labels_))

**Markdown answers — Part 6**
- (a) The five segment names and actions are: Low Income/High Spending — offer affordable loyalty rewards; Low Income/Low Spending — send low-cost introductory offers; Mid Income/Mid Spending — recommend popular products/bundles; High Income/High Spending — offer premium products/VIP rewards; High Income/Low Spending — send personalized recommendations or a limited trial offer.
- (b) High Income/High Spending is valuable now because customers have both high purchasing power and high spending. High Income/Low Spending is a major opportunity because purchasing power is high but current spending is low.
- (c) Gender does not strongly separate the groups here: female shares are fairly close (roughly 46–61%) and overall female share is 56%.
- (d) Adding Age changes the grouping; choose the segmentation that is most useful, interpretable and stable for the business question. Do not compare silhouette scores across different feature sets as if they were on the same scale.

# Part 7 — Final Output and Wrap-Up
## Task 7.1 — Save the segments
Create the required CSV and write the executive summary.

In [ ]:
final_output = seg_df[['CustomerID', 'Age', 'Gender', 'Income', 'Spending', 'Cluster', 'Segment']].copy()
final_output.to_csv('customer_segments.csv', index=False)
print('Saved customer_segments.csv')
print('Number of customers per segment:')
display(final_output['Segment'].value_counts().rename_axis('Segment').reset_index(name='Customers'))

### Executive Summary (6–8 lines)
1. K=5 was selected for Income and Spending because it gave the highest average silhouette (about 0.554) and the elbow curve bent around K=5.
2. The five segments were Low Income/High Spending (22), Low Income/Low Spending (23), Mid Income/Mid Spending (81), High Income/High Spending (39), and High Income/Low Spending (35).
3. Use affordable loyalty rewards for low-income/high-spending customers and introductory offers for low-income/low-spending customers.
4. Recommend bundles to the mid-income/mid-spending group, premium/VIP offers to high-income/high-spending customers, and personalized offers to high-income/low-spending customers.
5. Scaling matters: changing Income units changed raw clusters, while StandardScaler made the unit conversion invariant.
6. The added extreme customer was detected by the mean + 3 standard deviations distance rule; four real customers were also flagged for review.
7. Limitation: K-Means assumes distance-based, roughly compact clusters and the segment names depend on the chosen features, scaling and K.

**Wrap-up answers**
- (a) A surprising result was that a single extreme customer changed the clustering substantially and produced a singleton cluster.
- (b) With one more hour, test cluster stability on bootstrap samples, compare alternative clustering methods, and validate whether the segments lead to useful marketing outcomes.